# Setup

In [ ]:
import glob
import matplotlib.pyplot as plt
from skimage import exposure, io, util

import pandas as pd
import polars as pl
import os
import json

In [ ]:
input  = '/content/drive/MyDrive/Vision/Dataset/1_coins/'
output = '/content/drive/MyDrive/Vision/Prepared/1_coins/'
csv_file = output + 'labels.csv'
metadata_file = output + 'metadata.yaml'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Image conversion
- color to grayscale
- JPG to PNG

In [ ]:
files = glob.glob(input + '*.jpg')

In [ ]:
for file in files:
  image = io.imread(file, as_gray=True)
  image = util.img_as_ubyte(image)
  filename = file.split('/')[-1].replace('.jpg', '.png')
  io.imsave(output + filename, image)
  print('saved as', filename)

In [ ]:
plt.imshow(image, cmap='gray')
plt.show()

In [ ]:
files = glob.glob(output + "*.png")
len(files)

# Metadata consolidation
- read each JSON file and add it as a row into a single CSV

**labels.csv**

    name;labels
    "5_1477145436";"5"
    ...
    "5_1477145499";"10,25"

In [ ]:
files_json = glob.glob(input + "*.json")

In [ ]:
names = []
labels = []
real_count = []

for file in files_json:
    try:
        # load json file
        with open(file, 'r') as f:
            data = json.load(f)

        file_name = os.path.basename(file).replace('.json', '')
        names.append(file_name)

        shapes = data.get('shapes', [])

        # creates a list of all json labels
        current_labels = [shape.get('label', '') for shape in shapes]

        # labels concatenated in a str with separator = ,
        labels_str = ",".join(current_labels)
        labels.append(labels_str)

        # count the number of labels (coin) in the image
        real_count.append(len(current_labels))

    except Exception as e:
        print(f"Erro ao processar o arquivo {file}: {e}")

df = pl.DataFrame({
    'name': names,
    'labels': labels,
    'real_count': real_count
})

csv_file = 'labels.csv'
df.write_csv(output + csv_file)

# Creating metadata.yaml

`metadata.yaml` - https://github.com/crdoconnor/strictyaml

COUNTING
- total instances
- counts min, max, mean, standard deviation, median
- counts distribution
- counts histogram (notebook only)

CLASSIFICATION
- label distribution, mode and percentages
- label histogram (notebook only)

In [ ]:
! pip install strictyaml

In [ ]:
from strictyaml import Map, Seq, Str, Int, Float, as_document, load

In [ ]:
df = pl.read_csv(csv_file)

In [ ]:
# COUNTING
total_instances = df.height
num_labels_per_image = df.with_columns(
    pl.col("labels").str.split(",").list.len().alias("n_labels")
)["n_labels"]

num_labels_per_image_min = num_labels_per_image.min()
num_labels_per_image_max = num_labels_per_image.max()
num_labels_per_image_mean = num_labels_per_image.mean()
num_labels_per_image_median = num_labels_per_image.median()
num_labels_per_image_std = num_labels_per_image.std()

counts_distribution = (
    num_labels_per_image.to_frame("n_labels")
    .select(pl.col("n_labels").value_counts(sort=False))
    .unnest("n_labels")
    .rename({"n_labels": "labels"})
    .sort("labels")
)

# CLASSIFICATION
all_labels = df.select(
    pl.col("labels").str.split(",")
).explode("labels")

label_distribution_counts = (
    all_labels.select(
        pl.col("labels").value_counts(sort=True)
    )
    .unnest("labels")
)

label_distribution_percent = (
    all_labels.select(
        pl.col("labels").value_counts(normalize=True, sort=True)
    )
    .unnest("labels")
)

label_mode = label_distribution_counts.head(1).select("labels").item()

In [ ]:
# Counts Histogram
plt.figure(figsize=(7, 4))
plt.bar(counts_distribution["labels"], counts_distribution["count"])
plt.title("Counts Histogram (Distribution of each Labels per Image)")
plt.xlabel("Number of Labels")
plt.ylabel("Frequency (Images)")

#  Label Histogram
plt.figure(figsize=(7, 4))
plt.bar(label_distribution_counts["labels"], label_distribution_counts["count"])
plt.title("Label Histogram (Frequency of Each Label)")
plt.xlabel("Label")
plt.ylabel("Frequency (Total)")
plt.show()

In [ ]:
schema = Map({
    "COUNTING": Map({
        "total_instances": Int(),
        "counts": Map({
            "min": Float(),
            "max": Float(),
            "mean": Float(),
            "median": Float(),
            "std": Float(),
        }),
        "counts_distribution": Seq(Map({"labels": Int(), "count": Int()})),
    }),
    "CLASSIFICATION": Map({
        "label_distribution_counts": Seq(Map({"labels": Str(), "count": Int()})),
        "label_distribution_percent": Seq(Map({"labels": Str(), "proportion": Float()})),
        "label_mode": Str(),
    }),
})

data = {
    "COUNTING": {
        "total_instances": int(total_instances),
        "counts": {
            "min": float(num_labels_per_image_min),
            "max": float(num_labels_per_image_max),
            "mean": float(num_labels_per_image_mean),
            "median": float(num_labels_per_image_median),
            "std": float(num_labels_per_image_std),
        },
        "counts_distribution": [
            {"labels": int(r["labels"]), "count": int(r["count"])}
            for r in counts_distribution.iter_rows(named=True)
        ],
    },
    "CLASSIFICATION": {
        "label_distribution_counts": [
            {"labels": r["labels"], "count": int(r["count"])}
            for r in label_distribution_counts.iter_rows(named=True)
        ],
        "label_distribution_percent": [
            {"labels": r["labels"], "proportion": float(r["proportion"])}
            for r in label_distribution_percent.iter_rows(named=True)
        ],
        "label_mode": str(label_mode) if label_mode is not None else "",
    },
}

doc = as_document(data, schema)
with open(metadata_file, "w", encoding="utf-8") as f:
    f.write(doc.as_yaml())


In [ ]:
print("--------COUNTING--------\n")
print(f"counting min:\n {num_labels_per_image_min}\n")
print(f"counting max:\n {num_labels_per_image_max}\n")
print(f"counting mean:\n {num_labels_per_image_mean}\n")
print(f"counting median:\n {num_labels_per_image_median}\n")
print(f"counting standard deviation:\n {num_labels_per_image_std}\n")

print(f"counting distribution:\n {counts_distribution}\n")

print("--------CLASSIFICATION--------\n")

print(f"labels distribution:\n {label_distribution_counts}\n")

print(f"labels distribution by percentage:\n {label_distribution_percent}\n")

print(f"labels mode: {label_mode}\n")

# Reading Metadata.yaml

In [ ]:
with open(metadata_file, "r", encoding="utf-8") as f:
    metadata = load(f.read(), schema)

print(metadata.as_yaml())
